<a href="https://colab.research.google.com/github/47MAK/Cynbit-LeadIQ/blob/main/Cynbit_Lead.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cynbit LeadIQ — AI-Powered Inquiry Intelligence & Lead Routing

**Proof of Concept | Google Colab**

## Objective
Demonstrate an internal AI-assisted workflow that processes sample inquiries from multiple existing entry points:
- Blueprint / project inquiry
- Service inquiry
- Product demo request
- General contact

The prototype classifies the inquiry, extracts available details, assigns an explainable priority, recommends a team and next action, and drafts a follow-up for human review.

> **Scope and safety:** This notebook uses synthetic sample inquiries only. It does not connect to Cynbit's live website, CRM, email, or customer database. Priority scores are heuristic workflow indicators, not validated probabilities of conversion. Human review is required.



## Step 1 — Install dependencies

In [ ]:
!pip -q install pandas scikit-learn sentence-transformers gradio

## Step 2 — Import libraries and load sample inquiry data

In [ ]:
import re
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

DATA_PATH = Path("/content/sample_inquiries.csv")

# The file is included in the project ZIP. Upload it to the Colab session if needed.
if not DATA_PATH.exists():
    from google.colab import files
    uploaded = files.upload()  # Upload data/sample_inquiries.csv
    DATA_PATH = Path(next(iter(uploaded.keys())))

df = pd.read_csv(DATA_PATH).fillna("")
print(f"Loaded {len(df)} synthetic inquiries")
display(df.head())

Loaded 16 synthetic inquiries


,inquiry_id,source_channel,inquiry_text,provided_category,industry,budget_inr,timeline,product
0,LQ001,Blueprint,We are a healthcare startup looking to build a...,HealthTech,Healthcare,500000.0,2 months,
1,LQ002,Service Inquiry,Need a mobile app for our food delivery busine...,Custom Software Development,Food & Beverage,800000.0,1 month,
2,LQ003,Product Demo,I would like to see a demo of TenantEase for m...,TenantEase,Real Estate,,Not specified,TenantEase
3,LQ004,General Contact,"Hello, I would like to know more about your co...",General,Other,,Not specified,
4,LQ005,Service Inquiry,We need help improving our cloud infrastructur...,Cloud & DevOps,Technology,,1 week,


## Step 3 — Define service taxonomy, channel labels, and routing rules

In [ ]:
CHANNEL_LABELS = [
    "Blueprint project inquiry",
    "Service inquiry",
    "Product demo request",
    "General contact or partnership inquiry"
]

SERVICE_LABELS = [
    "AI Automation & Agents",
    "Custom Software Development",
    "Cloud & DevOps",
    "CRM & Sales Automation",
    "Digital Marketing & SEO",
    "Cynbit Studio",
    "TenantEase",
    "Revio",
    "General / Needs Discovery"
]

ROUTING = {
    "AI Automation & Agents": ("AI Solutions Team", "Review automation requirements and arrange a discovery call"),
    "Custom Software Development": ("Software Engineering Team", "Review scope and identify technical requirements"),
    "Cloud & DevOps": ("Cloud & DevOps Team", "Arrange a technical discovery call"),
    "CRM & Sales Automation": ("Business Automation Team", "Review current CRM workflow and integration needs"),
    "Digital Marketing & SEO": ("Digital Growth Team", "Review website and marketing objectives"),
    "Cynbit Studio": ("Product Team — Cynbit Studio", "Arrange a product walkthrough"),
    "TenantEase": ("Product Team — TenantEase", "Arrange a product walkthrough"),
    "Revio": ("Product Team — Revio", "Arrange a product walkthrough"),
    "General / Needs Discovery": ("Business Development Team", "Clarify the inquiry and identify the right point of contact")
}

## Step 4 — Build the NLP classifier

The notebook first attempts to use the lightweight Sentence Transformers model `all-MiniLM-L6-v2`. It compares inquiry text with short category descriptions using cosine similarity.

If the model cannot be downloaded or initialized, the notebook automatically falls back to TF-IDF. This makes the demo more resilient in environments with limited model access.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

SERVICE_DESCRIPTIONS = {
    "AI Automation & Agents": "AI automation, LLM applications, autonomous agents, chatbots, voice agents, workflow automation",
    "Custom Software Development": "custom software, web application, mobile app, SaaS platform, website and application development",
    "Cloud & DevOps": "cloud infrastructure, cloud migration, DevOps, CI CD, deployment, reliability and infrastructure",
    "CRM & Sales Automation": "CRM integration, sales automation, lead assignment, sales workflows and reporting",
    "Digital Marketing & SEO": "digital marketing, search engine optimization, technical SEO, ecommerce marketing",
    "Cynbit Studio": "Cynbit Studio product demo and product information",
    "TenantEase": "TenantEase property rental and property management product demo",
    "Revio": "Revio product demo and sales platform information",
    "General / Needs Discovery": "general question, partnership, collaboration or unclear request"
}

CHANNEL_DESCRIPTIONS = {
    "Blueprint": "A broad project or solution idea submitted through the Tell us what you are building Blueprint inquiry",
    "Service Inquiry": "A request about a specific service such as AI, software development, cloud, CRM or SEO",
    "Product Demo": "A request to schedule a demonstration or learn about a Cynbit product",
    "General Contact": "A general contact, partnership or other message"
}

embedder = None
classifier_mode = "TF-IDF fallback"

try:
    from sentence_transformers import SentenceTransformer
    embedder = SentenceTransformer("all-MiniLM-L6-v2")
    classifier_mode = "Sentence Transformers embeddings"
except Exception as e:
    print("Embedding model unavailable; using TF-IDF fallback.")
    print("Reason:", str(e)[:250])

def best_match(text, descriptions):
    labels = list(descriptions.keys())
    docs = [descriptions[k] for k in labels]
    if embedder is not None:
        vectors = embedder.encode([str(text)] + docs, normalize_embeddings=True)
        scores = cosine_similarity([vectors[0]], vectors[1:])[0]
    else:
        vectorizer = TfidfVectorizer(ngram_range=(1,2), stop_words="english")
        matrix = vectorizer.fit_transform([str(text)] + docs)
        scores = cosine_similarity(matrix[0:1], matrix[1:])[0]
    idx = int(np.argmax(scores))
    return labels[idx], float(scores[idx])

print("Classifier mode:", classifier_mode)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Classifier mode: Sentence Transformers embeddings


## Step 5 — Extract details from inquiry text

In [ ]:
def extract_budget(text, provided_budget=""):
    if str(provided_budget).strip():
        try:
            return int(float(str(provided_budget).replace(",", "").strip()))
        except:
            pass

    t = str(text).lower().replace(",", "")
    # Examples: Rs 500000, INR 2 lakh, 5 lakhs, 3.5 crore
    patterns = [
        (r'(?:rs\.?|inr|₹)\s*(\d+(?:\.\d+)?)\s*(crore|cr|lakh| lakhs|lac|lacs|k)?', 1),
        (r'\b(\d+(?:\.\d+)?)\s*(crore|cr|lakh|lakhs|lac|lacs)\b', 1)
    ]
    for pattern, _ in patterns:
        m = re.search(pattern, t)
        if m:
            amount = float(m.group(1))
            unit = (m.group(2) or "").strip()
            multiplier = 1
            if unit in ["lakh", "lakhs", "lac", "lacs"]:
                multiplier = 100000
            elif unit in ["crore", "cr"]:
                multiplier = 10000000
            elif unit == "k":
                multiplier = 1000
            return int(amount * multiplier)
    return None

def extract_timeline(text, provided_timeline=""):
    if str(provided_timeline).strip() and str(provided_timeline).lower() not in ["not specified", "nan"]:
        return str(provided_timeline)
    t = str(text).lower()
    patterns = [
        r'\bwithin\s+\d+\s+(?:days?|weeks?|months?)\b',
        r'\b\d+\s+(?:days?|weeks?|months?)\b',
        r'\bnext\s+(?:week|month)\b',
        r'\b(?:urgent|as soon as possible|immediately|later this year)\b'
    ]
    for p in patterns:
        m = re.search(p, t)
        if m:
            return m.group(0)
    return "Not provided"

def extract_company_context(text, provided_industry=""):
    return str(provided_industry).strip() if str(provided_industry).strip() else "Not provided"

## Step 6 — Priority scoring and routing logic

In [ ]:
def calculate_priority(text, budget, timeline, source_channel):
    t = str(text).lower()
    score = 0
    reasons = []

    # Budget signal: presence only; higher budget is not automatically better.
    if budget is not None:
        score += 20
        reasons.append("Budget information provided (+20)")

    # Timeline / urgency signal
    if any(word in t for word in ["urgent", "as soon as possible", "immediately", "this week"]):
        score += 25
        reasons.append("Urgent timing mentioned (+25)")
    elif timeline != "Not provided" and timeline != "Not specified":
        score += 15
        reasons.append("Timeline provided (+15)")

    # Intent signal
    if any(word in t for word in ["schedule", "demo", "proposal", "start", "begin", "looking to build", "need help", "need a"]):
        score += 20
        reasons.append("Action-oriented intent detected (+20)")

    # Requirement completeness
    detail_terms = ["app", "platform", "automation", "integration", "dashboard", "cloud", "crm", "website", "agent", "pipeline"]
    matched = sum(term in t for term in detail_terms)
    if matched >= 3:
        score += 25
        reasons.append("Detailed requirements detected (+25)")
    elif matched >= 1:
        score += 15
        reasons.append("Some requirements detected (+15)")
    else:
        score += 5
        reasons.append("Limited requirement detail (+5)")

    # Channel context: demo requests indicate explicit product interest
    if source_channel == "Product Demo":
        score += 10
        reasons.append("Explicit product demo interest (+10)")

    score = min(score, 100)
    if score >= 70:
        priority = "High"
    elif score >= 40:
        priority = "Medium"
    else:
        priority = "Low"
    return score, priority, reasons

def recommend_followup(category, priority, timeline):
    team, action = ROUTING.get(category, ROUTING["General / Needs Discovery"])
    if priority == "High":
        opening = "Thank you for sharing your requirements. Our team will prioritize reviewing your inquiry."
    elif priority == "Medium":
        opening = "Thank you for contacting Cynbit. We would be happy to understand your requirements in more detail."
    else:
        opening = "Thank you for your interest in Cynbit. We would be glad to learn more about what you are looking to achieve."
    return team, action, f"{opening} A member of our team can follow up to discuss the next steps. (Suggested draft — please review before sending.)"

## Step 7 — Create the end-to-end inquiry analysis function

In [ ]:
def analyze_inquiry(inquiry_text, source_channel="Blueprint", industry="", budget="", timeline="", product=""):
    text = str(inquiry_text).strip()
    if not text:
        raise ValueError("Please enter an inquiry description.")

    # Classify service/product from the text, while honoring an explicit product field.
    category, confidence = best_match(text, SERVICE_DESCRIPTIONS)
    if str(product).strip() in ["Cynbit Studio", "TenantEase", "Revio"]:
        category = str(product).strip()

    extracted_budget = extract_budget(text, budget)
    extracted_timeline = extract_timeline(text, timeline)
    industry_value = extract_company_context(text, industry)

    score, priority, reasons = calculate_priority(
        text, extracted_budget, extracted_timeline, source_channel
    )
    team, next_action, followup = recommend_followup(category, priority, extracted_timeline)

    return {
        "Source channel": source_channel,
        "Predicted category": category,
        "Model similarity score": round(confidence, 3),
        "Industry": industry_value,
        "Budget (INR)": extracted_budget if extracted_budget is not None else "Not provided",
        "Timeline": extracted_timeline,
        "Priority score": score,
        "Priority": priority,
        "Scoring explanation": "; ".join(reasons),
        "Recommended team": team,
        "Suggested next action": next_action,
        "Follow-up draft": followup
    }

demo = analyze_inquiry(
    "We need an AI voice agent to handle inbound calls and qualify customer requests. Please contact us urgently.",
    source_channel="Service Inquiry",
    industry="Professional Services"
)
pd.DataFrame([demo]).T.rename(columns={0:"Result"})

## Step 8 — Process the complete sample dataset

In [ ]:
results = []
for _, row in df.iterrows():
    result = analyze_inquiry(
        inquiry_text=row["inquiry_text"],
        source_channel=row["source_channel"],
        industry=row.get("industry", ""),
        budget=row.get("budget_inr", ""),
        timeline=row.get("timeline", ""),
        product=row.get("product", "")
    )
    result["Inquiry ID"] = row["inquiry_id"]
    result["Original inquiry"] = row["inquiry_text"]
    results.append(result)

results_df = pd.DataFrame(results)
display(results_df[[
    "Inquiry ID", "Source channel", "Predicted category", "Priority score",
    "Priority", "Recommended team", "Suggested next action"
]])

## Step 9 — Basic results dashboard

In [ ]:
import matplotlib.pyplot as plt

priority_counts = results_df["Priority"].value_counts().reindex(["High","Medium","Low"], fill_value=0)
plt.figure(figsize=(6,4))
priority_counts.plot(kind="bar")
plt.title("Sample inquiries by priority")
plt.xlabel("Priority")
plt.ylabel("Number of inquiries")
plt.xticks(rotation=0)
plt.show()

channel_counts = results_df["Source channel"].value_counts()
plt.figure(figsize=(7,4))
channel_counts.plot(kind="bar")
plt.title("Inquiries by source channel")
plt.xlabel("Source channel")
plt.ylabel("Number of inquiries")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

print("Total inquiries:", len(results_df))
print("High priority:", int(priority_counts["High"]))
print("Medium priority:", int(priority_counts["Medium"]))
print("Low priority:", int(priority_counts["Low"]))

## Step 10 — Interactive demo using Gradio

In [ ]:
import gradio as gr

def run_demo(inquiry_text, source_channel, industry, budget, timeline, product):
    try:
        result = analyze_inquiry(inquiry_text, source_channel, industry, budget, timeline, product)
        return pd.DataFrame({"Field": list(result.keys()), "AI-assisted output": list(result.values())})
    except Exception as e:
        return pd.DataFrame({"Error": [str(e)]})

with gr.Blocks(title="Cynbit LeadIQ — Inquiry Intelligence") as app:
    gr.Markdown("# Cynbit LeadIQ")
    gr.Markdown("AI-assisted inquiry classification, prioritization and routing — synthetic-data prototype.")
    inquiry = gr.Textbox(label="Customer inquiry", lines=5, placeholder="Describe the customer's requirement...")
    channel = gr.Dropdown(
        ["Blueprint", "Service Inquiry", "Product Demo", "General Contact"],
        value="Blueprint", label="Existing inquiry channel"
    )
    industry = gr.Textbox(label="Industry (optional)")
    budget = gr.Textbox(label="Budget in INR (optional)", placeholder="e.g. 500000")
    timeline = gr.Textbox(label="Timeline (optional)", placeholder="e.g. 2 months")
    product = gr.Dropdown(["", "Cynbit Studio", "TenantEase", "Revio"], value="", label="Product (if demo request)")
    submit = gr.Button("Analyze inquiry")
    output = gr.Dataframe(label="Inquiry intelligence report", interactive=False)
    submit.click(run_demo, inputs=[inquiry, channel, industry, budget, timeline, product], outputs=output)

app.launch(share=True)

## Step 11 — Export analyzed results

In [ ]:
output_path = "/content/leadiq_analyzed_inquiries.csv"
results_df.to_csv(output_path, index=False)
print("Saved:", output_path)
from google.colab import files
files.download(output_path)

## Step 12 — Evaluation, limitations, and next steps

### What this prototype demonstrates
- Accepts inquiries from four simulated existing channels.
- Uses semantic similarity (or TF-IDF fallback) to suggest a service/product category.
- Extracts budget and timeline when provided.
- Calculates a transparent rule-based priority score.
- Recommends a team and next action.
- Drafts a follow-up message for human review.
- Displays results in a simple Gradio interface.

### How to evaluate it
1. Run the notebook from top to bottom.
2. Try at least five new inquiries with different channels.
3. Check whether the predicted category and routing recommendation make sense.
4. Verify extracted budget and timeline against the original text.
5. Note errors and adjust category descriptions or extraction rules.
6. Do not report model accuracy unless you create a labeled test set and calculate metrics against its ground-truth labels.

### Current limitations
- Sample data is synthetic and small.
- Service classification is a prototype, not a trained Cynbit-specific model.
- Budget/timeline extraction uses simple patterns and may miss unusual formats.
- Priority is a heuristic score, not conversion probability.
- No live form, CRM, email, or database integration.
- Follow-up drafts are not sent automatically.

### Future enhancements
- Connect approved form submissions through secure APIs/webhooks.
- Integrate with Cynbit's existing CRM after workflow discovery.
- Add duplicate inquiry detection and conversation history.
- Use an LLM with structured output for richer requirement extraction.
- Train and validate lead conversion prediction using historical outcomes.
- Add role-based dashboard, audit logs, and feedback-based model improvement.
- Add human approval before routing or sending any communication.

### Suggested demo explanation
"This proof of concept demonstrates an AI-assisted layer that can sit behind existing inquiry channels. It takes a sample inquiry, identifies its likely category, extracts available details, assigns an explainable priority, and recommends a team and next action. I used synthetic data because I do not have access to Cynbit's private inquiry or CRM data. The next step would be to validate the workflow with the team and explore secure integration with the systems already in use."